# LLM Conversation Keywords Extractor

## Function Definition

In [3]:
import json
import spacy
from collections import Counter
import unicodedata
from typing import List
from role_enum import Role

nlp = spacy.load("en_core_web_sm", disable=["ner"])

def normalize(text: str) -> str:
    text = text.lower()
    text = unicodedata.normalize("NFKD", text).encode("ascii", "ignore").decode("utf-8")
    return text


def extract_top_keywords(
    json_path: str,
    roles: List[Role],
    top_n: int = 20,
    lemmatize: bool = False
):
    with open(json_path, "r", encoding="utf-8") as f:
        data = json.load(f)

    texts = []
    messages = data.get("messages", [])

    allowed_roles = {r.value for r in roles}

    for msg in messages:
        if msg.get("role") in allowed_roles:
            content = msg.get("content", "")
            if content:
                texts.append(normalize(content))

    all_tokens = []

    for doc in nlp.pipe(texts):
        for token in doc:
            if (
                not token.is_stop and
                not token.is_punct and
                token.is_alpha and
                len(token.text) > 2
            ):
                word = token.lemma_.lower() if lemmatize else token.text.lower()
                all_tokens.append(word)

    counter = Counter(all_tokens)

    result = [
        {"keyword": word, "frequency": freq}
        for word, freq in counter.most_common(top_n)
    ]

    return result

## Usage

In [5]:
keywords = extract_top_keywords("data/sample_healthcare.json", roles=[Role.USER, Role.ASSISTANT], top_n=30, lemmatize=False)

display(keywords)

[{'keyword': 'diagnosis', 'frequency': 41},
 {'keyword': 'disease', 'frequency': 36},
 {'keyword': 'blood', 'frequency': 25},
 {'keyword': 'case', 'frequency': 24},
 {'keyword': 'year', 'frequency': 22},
 {'keyword': 'old', 'frequency': 22},
 {'keyword': 'treatment', 'frequency': 21},
 {'keyword': 'clinical', 'frequency': 20},
 {'keyword': 'patient', 'frequency': 20},
 {'keyword': 'therapy', 'frequency': 20},
 {'keyword': 'pain', 'frequency': 16},
 {'keyword': 'biopsy', 'frequency': 15},
 {'keyword': 'chest', 'frequency': 14},
 {'keyword': 'management', 'frequency': 14},
 {'keyword': 'glucose', 'frequency': 14},
 {'keyword': 'include', 'frequency': 13},
 {'keyword': 'shows', 'frequency': 13},
 {'keyword': 'count', 'frequency': 13},
 {'keyword': 'based', 'frequency': 13},
 {'keyword': 'stroke', 'frequency': 13},
 {'keyword': 'acute', 'frequency': 12},
 {'keyword': 'includes', 'frequency': 12},
 {'keyword': 'pressure', 'frequency': 12},
 {'keyword': 'anti', 'frequency': 12},
 {'keyword':